In [ ]:
# # T4 — RoBERTa Dynabench (Kaggle Edition)
# ## AudioGuard FYP | TCA Track
#
# **Purpose**: Fine-tune `facebook/roberta-hate-speech-dynabench-r4-target` for **binary**
# hate speech classification on Davidson. Labels are remapped to binary (hate=0 / not-hate=1)
# **before** splitting to avoid data leakage. This model was trained adversarially using
# Dynabench annotation rounds.
#
# ---
# ### ⚙️ Kaggle Setup Instructions
# 1. Enable GPU: **Settings** → **Accelerator** → **GPU T4 x2** (recommended)
# 2. Enable **Internet**: **Settings** → **Internet** → On
#    *(Required to download the model and Davidson dataset from HuggingFace)*
# 3. No manual dataset upload needed — Davidson is pulled from HuggingFace automatically
#
# **Expected runtime**: ~8–15 min on Kaggle GPU T4

In [ ]:
# Cell 2 — Install missing dependencies
import subprocess, sys
 
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
 
for pkg, import_name in [('transformers', 'transformers'),
                          ('datasets',     'datasets'),
                          ('accelerate',   'accelerate')]:
    try:
        __import__(import_name)
        print(f'✓ {pkg} already installed')
    except ImportError:
        print(f'Installing {pkg}...')
        pip_install(pkg)
 
print('✓ Dependencies ready')

In [ ]:
# Cell 3 — Imports
import os, sys, json, time, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm  # standard tqdm — more stable on Kaggle than tqdm.notebook
 
print('✓ All imports successful')

In [ ]:
# Cell 4 — Environment check
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {name} | VRAM: {vram:.1f} GB')
else:
    print('⚠️  No GPU detected — go to Settings → Accelerator → GPU T4 x2')
 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')

In [ ]:
# Cell 5 — Reproducibility seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f'Seeds set to {SEED}')

In [ ]:
# Cell 6 — CONFIG
# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT_DIR: Kaggle persists /kaggle/working/ after the session ends.
# Download outputs via the Output tab on the right panel.
# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
CONFIG = {
    'model_id':      'T4',
    'model_name':    'facebook/roberta-hate-speech-dynabench-r4-target',
    'track':         'TCA',
    'num_labels':    2,        # binary: hate=0 / not-hate=1
    'epochs':        10,       # max epochs — early stopping fires before this if needed
    'batch_size':    32,
    'learning_rate': 2e-5,
    'warmup_ratio':  0.1,
    'weight_decay':  0.01,
    'max_length':    128,
    'seed':          42,
    'output_dir':    OUTPUT_DIR,
    'dataset':       '/kaggle/input/datasets/eldrich/hate-speech-offensive-tweets-by-davidson-et-al',
    'device':        device,
    # ── Early stopping ──────────────────────────────────────────────────────
    'early_stopping': True,   # set False to always run all epochs
    'patience':       3,      # stop after N epochs with no val F1 improvement
    'min_delta':      0.001,  # minimum improvement to count as "better"
}
 
print('CONFIG loaded:', json.dumps({k: str(v) for k, v in CONFIG.items()}, indent=2))

In [ ]:
# Cell 6b — Auto-adjust batch size based on VRAM
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb < 6:
        CONFIG['batch_size'] = max(2, CONFIG['batch_size'] // 8)
        CONFIG['gradient_accumulation_steps'] = 8
        print(f'Auto-adjusted for <6 GB VRAM: batch={CONFIG["batch_size"]}, grad_accum=8')
    elif vram_gb < 10:
        CONFIG['batch_size'] = max(4, CONFIG['batch_size'] // 4)
        CONFIG['gradient_accumulation_steps'] = 4
        print(f'Auto-adjusted for <10 GB VRAM: batch={CONFIG["batch_size"]}, grad_accum=4')
    else:
        CONFIG['gradient_accumulation_steps'] = 1
        print(f'VRAM {vram_gb:.1f} GB — no batch size adjustment needed')
else:
    CONFIG['gradient_accumulation_steps'] = 1
    print('CPU mode — using configured batch size')

In [ ]:
# Cell 8 — Load Davidson dataset and remap to binary BEFORE split
# ─────────────────────────────────────────────────────────────────────────────
# CRITICAL: binary remapping MUST happen BEFORE train/val/test split to prevent
# data leakage. Splitting first, then remapping per-split would cause leakage.
# ─────────────────────────────────────────────────────────────────────────────
print('Loading Davidson hate speech dataset from HuggingFace...')
try:
    ds = load_dataset('tdavidson/hate_speech_offensive')
    df = ds['train'].to_pandas()
    print(f'✓ Davidson loaded: {len(df):,} rows')
    print(f'Raw label dist:\n{df["class"].value_counts()}')
except Exception as e:
    print(f'✗ Dataset loading failed: {e}')
    print('Fix: Make sure Internet is ON in Kaggle Settings.')
    raise
 
# Remap to binary: 0=hate, 1=offensive→not-hate, 2=neither→not-hate
BINARY_MAP = {0: 0, 1: 1, 2: 1}
df['label_binary'] = df['class'].map(BINARY_MAP)
print(f'\nBinary label dist:\n{df["label_binary"].value_counts()}')
print('  0 = hate  |  1 = not-hate')

In [ ]:
# Cell 9 — Preprocessing + Split
def clean_tweet(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)        # remove URLs
    text = re.sub(r'@\w+', '@user', text)      # normalise mentions
    return re.sub(r'\s+', ' ', text).strip()
 
print('Before:', df['tweet'].iloc[0])
df['text_clean'] = df['tweet'].apply(clean_tweet)
print('After: ', df['text_clean'].iloc[0])
 
# Train / Val / Test split  (80 / 10 / 10)
train_df, temp_df = train_test_split(df,      test_size=0.20, random_state=SEED, stratify=df['label_binary'])
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label_binary'])
print(f'\nTrain: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Train label dist: {dict(train_df["label_binary"].value_counts())}')

In [ ]:
# Cell 9b — PyTorch Dataset & DataLoaders
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
 
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = list(texts)
        self.labels     = list(labels)
        self.tok        = tokenizer
        self.max_length = max_length
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }
 
num_workers = 2 if device == 'cuda' else 0
 
train_ds = TweetDataset(train_df['text_clean'], train_df['label_binary'], tokenizer, CONFIG['max_length'])
val_ds   = TweetDataset(val_df['text_clean'],   val_df['label_binary'],   tokenizer, CONFIG['max_length'])
test_ds  = TweetDataset(test_df['text_clean'],  test_df['label_binary'],  tokenizer, CONFIG['max_length'])
 
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=num_workers, pin_memory=(device=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'] * 2,
                          shuffle=False, num_workers=num_workers, pin_memory=(device=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'] * 2,
                          shuffle=False, num_workers=num_workers, pin_memory=(device=='cuda'))
 
print(f'✓ DataLoaders ready: {len(train_loader)} train | {len(val_loader)} val | {len(test_loader)} test batches')


In [ ]:
# Cell 10 — Load pretrained RoBERTa Dynabench model
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=CONFIG['num_labels'],
        ignore_mismatched_sizes=True   # safe — model was pretrained with 2 labels already
    )
    model = model.to(device)
    print(f'✓ Model loaded: {CONFIG["model_name"]}')
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Total parameters:     {total_params:,}')
    print(f'  Trainable parameters: {trainable_params:,}')
except Exception as e:
    print(f'✗ Model loading failed: {e}')
    print('Fix: Ensure Internet is ON in Kaggle Settings.')
    raise
 
# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint resume — set RESUME = True to load from a previous saved checkpoint.
# (input() is not supported on Kaggle, so we use a flag instead.)
# ─────────────────────────────────────────────────────────────────────────────
RESUME = False   # ← flip to True to resume from checkpoint
checkpoint_path = os.path.join(CONFIG['output_dir'], 'best_model')
 
if RESUME and os.path.exists(checkpoint_path):
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path).to(device)
    print(f'✓ Resumed from checkpoint: {checkpoint_path}')
elif RESUME:
    print(f'⚠️  RESUME=True but no checkpoint found at {checkpoint_path} — starting fresh')
else:
    print('Starting fresh training (RESUME=False)')

In [ ]:
# Cell 10b — CPU/GPU notice
if device == 'cpu':
    print('⚠️  Training on CPU. T4 Davidson 24k will take 2–3 hours.')
    print('    Enable GPU: Settings → Accelerator → GPU T4 x2')
else:
    print(f'✓ Training on GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 11 — Training loop with early stopping
optimizer    = torch.optim.AdamW(model.parameters(),
                                  lr=CONFIG['learning_rate'],
                                  weight_decay=CONFIG['weight_decay'])
total_steps  = len(train_loader) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
 
grad_accum       = CONFIG.get('gradient_accumulation_steps', 1)
train_losses     = []
val_losses       = []
val_accuracies   = []
val_f1s          = []
best_val_f1      = 0.0
best_epoch       = 0
total_train_time = 0.0
 
# ── Early stopping state ───────────────────────────────────────────────────
patience_counter = 0
es_patience      = CONFIG['patience']
es_min_delta     = CONFIG['min_delta']
stopped_early    = False
 
print(f'Training for up to {CONFIG["epochs"]} epochs '
      f'(early stopping: patience={es_patience}, min_delta={es_min_delta})\n')
 
for epoch in range(1, CONFIG['epochs'] + 1):
    t0 = time.time()
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0
 
    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{CONFIG["epochs"]} [train]')):
        kwargs  = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**kwargs)
        loss    = outputs.loss / grad_accum
        loss.backward()
        running_loss += outputs.loss.item()
 
        if (step + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
 
    avg_train_loss = running_loss / len(train_loader)
 
    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    val_loss_sum          = 0.0
    all_preds, all_labels = [], []
 
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch}/{CONFIG["epochs"]} [val]  '):
            kwargs  = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**kwargs)
            val_loss_sum += outputs.loss.item()
            all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            all_labels.extend(kwargs['labels'].cpu().numpy())
 
    avg_val_loss = val_loss_sum / len(val_loader)
    val_acc      = accuracy_score(all_labels, all_preds)
    val_f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    epoch_time   = time.time() - t0
    total_train_time += epoch_time
    m, s = divmod(int(epoch_time), 60)
 
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)
    val_f1s.append(val_f1)
 
    # ── Checkpoint if improved ────────────────────────────────────────────────
    improved    = val_f1 > (best_val_f1 + es_min_delta)
    best_marker = ''
    if improved:
        best_val_f1      = val_f1
        best_epoch       = epoch
        patience_counter = 0
        model.save_pretrained(os.path.join(CONFIG['output_dir'], 'best_model'))
        tokenizer.save_pretrained(os.path.join(CONFIG['output_dir'], 'best_model'))
        best_marker = ' ← BEST'
    else:
        patience_counter += 1
        best_marker = f' (no improvement {patience_counter}/{es_patience})'
 
    print(
        f'Epoch {epoch}/{CONFIG["epochs"]} | '
        f'Train Loss: {avg_train_loss:.4f} | '
        f'Val Loss: {avg_val_loss:.4f} | '
        f'Val Acc: {val_acc*100:.2f}% | '
        f'Val F1: {val_f1:.4f} | '
        f'Time: {m}m {s}s{best_marker}'
    )
 
    # ── Early stopping check ──────────────────────────────────────────────────
    if CONFIG['early_stopping'] and patience_counter >= es_patience:
        print(f'\n⏹  Early stopping triggered — val F1 did not improve by >{es_min_delta} '
              f'for {es_patience} consecutive epochs.')
        stopped_early = True
        break
 
if not stopped_early:
    print(f'\n✓ Completed all {CONFIG["epochs"]} epochs.')
print(f'✓ Best epoch: {best_epoch} | Best Val F1: {best_val_f1:.4f}')


In [ ]:
# Cell 12 — Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss', color='royalblue',  marker='o')
ax1.plot(val_losses,   label='Val Loss',   color='darkorange', marker='o')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.set_xlabel('Epoch'); ax1.grid(True, alpha=0.3)
 
ax2.plot(val_accuracies, label='Val Accuracy', color='seagreen', marker='o')
ax2.plot(val_f1s,        label='Val F1 Macro', color='crimson',  marker='o')
ax2.set_title('Validation Metrics'); ax2.legend(); ax2.set_xlabel('Epoch'); ax2.grid(True, alpha=0.3)
 
# Mark best epoch when early stopping fired
if stopped_early:
    ax1.axvline(x=best_epoch - 1, color='grey', linestyle='--', alpha=0.6, label='Best epoch')
    ax2.axvline(x=best_epoch - 1, color='grey', linestyle='--', alpha=0.6, label='Best epoch')
 
plt.tight_layout()
plot_path = os.path.join(CONFIG['output_dir'], 'training_curves.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'✓ Training curves saved → {plot_path}')

In [ ]:
# Cell 13 — Test Set Evaluation
best_model_path = os.path.join(CONFIG['output_dir'], 'best_model')
model_best = AutoModelForSequenceClassification.from_pretrained(best_model_path).to(device)
model_best.eval()
 
all_preds, all_labels_test = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test evaluation'):
        labels  = batch.pop('labels')
        kwargs  = {k: v.to(device) for k, v in batch.items()}
        outputs = model_best(**kwargs)
        all_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        all_labels_test.extend(labels.numpy())
 
label_names    = ['hate', 'not-hate']
test_accuracy  = accuracy_score(all_labels_test, all_preds)
test_f1        = f1_score(all_labels_test,        all_preds, average='macro', zero_division=0)
test_precision = precision_score(all_labels_test, all_preds, average='macro', zero_division=0)
test_recall    = recall_score(all_labels_test,    all_preds, average='macro', zero_division=0)
 
print('\n=== TEST SET RESULTS ===')
print(classification_report(all_labels_test, all_preds, target_names=label_names, digits=4))
 
cm = confusion_matrix(all_labels_test, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_names,
            yticklabels=label_names, cmap='Blues')
plt.title('RoBERTa Dynabench — Confusion Matrix')
plt.tight_layout()
cm_path = os.path.join(CONFIG['output_dir'], 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'✓ Confusion matrix saved → {cm_path}')

In [ ]:
# Cell 14 — Save training_summary.json
peak_vram = (round(torch.cuda.max_memory_allocated() / 1e9, 2)
             if torch.cuda.is_available() else 0.0)
 
results = {
    'model_id':                CONFIG['model_id'],
    'model_name':              CONFIG['model_name'],
    'track':                   CONFIG['track'],
    'accuracy':                round(test_accuracy,  4),
    'f1_macro':                round(test_f1,        4),
    'precision_macro':         round(test_precision, 4),
    'recall_macro':            round(test_recall,    4),
    'train_time_minutes':      round(total_train_time / 60, 2),
    'peak_vram_gb':            peak_vram,
    'epochs_trained':          best_epoch,
    'max_epochs':              CONFIG['epochs'],
    'stopped_early':           stopped_early,
    'early_stopping_patience': es_patience,
    'binary_label_map':        '0=hate, 1=not-hate (offensive+neither merged)',
    'dataset':                 CONFIG['dataset'],
    'saved_model_path':        CONFIG['output_dir'],
    'timestamp':               time.strftime('%Y-%m-%d %H:%M:%S')
}
 
# Merge with existing summary (shared across T1/T2/T3/T4)
summary_path = '/kaggle/working/outputs/training_summary.json'
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
 
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    summary = [r for r in summary if r['model_id'] != CONFIG['model_id']]
else:
    summary = []
 
summary.append(results)
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
 
print(f'✓ Results saved → {summary_path}')
print(json.dumps(results, indent=2))
